# v0.4 S5-NQ — full-corpus validation of the NQ transfer ratios, passage-text checks (addendum v0.4)
**How to run:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)**. It runs unattended (about 3.5–4.5 h). If a step fails, the notebook stops with `FAILED` — send me the log. When it finishes, open the version's **Output** tab, download `v04_s5nq_results.zip` and send it to me.

**Input:** Add Input → **Your Work → Notebooks** → the v0.2 **Main S1** notebook from 27 Sep (its output has `main_s1_results.zip`), the same input as the S3 and S4 notebooks.

In [ ]:
# Cell 0 — helper: run a shell command, stream its output, STOP the notebook if it fails
import os, subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

In [ ]:
# Cell 0b — run two commands at once (one per GPU); STOP if either fails
import threading
def sh_par(*cmds):
    errs = []
    def run(c):
        try:
            sh(c)
        except Exception as e:
            errs.append(str(e))
    ts = [threading.Thread(target=run, args=(c,)) for c in cmds]
    [t.start() for t in ts]; [t.join() for t in ts]
    if errs:
        raise RuntimeError("; ".join(errs))

In [ ]:
# Cell 0d — a failing step is recorded and the notebook continues, so finished results are always packaged
GEN_ERR = ""
def soft(*cmds, par=False):
    global GEN_ERR
    try:
        sh_par(*cmds) if par else [sh(c) for c in cmds]
    except Exception as e:
        GEN_ERR += ("; " if GEN_ERR else "") + str(e); print("STEP FAILED (recorded, continuing):", e)

In [ ]:
# Cell 0c — the code of commit 80b647e as base64 tar.gz (do not edit; Cell 1 checks it)
CODE = "..."

In [ ]:
# Cell 1 — hardware, code frozen at 80b647e (embedded in Cell 0c), pinned environment, check, tests
sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv || true")
sh("df -h / /tmp /kaggle/working && free -g && nproc")
os.chdir("/kaggle/working")
try:                                   # Hugging Face token from Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception as e:
    print("HF_TOKEN not available (public downloads still work):", type(e).__name__)
# the GitHub repo is private: the code of commit 80b647ef079f8a650c60e6ddd3bd9bb6f14ed7c8 is embedded in Cell 0c (no dataset, no clone);
# STOP unless the extracted files reproduce that commit's git tree exactly (paper_inputs, paper_outputs, paper, paper_figures, manuscript_ipm, literature left out)
import base64, io, shutil, tarfile
shutil.rmtree("rag", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE))).extractall("rag")
os.chdir("/kaggle/working/rag")
sh("git init -q && git add -A -f")
sh("test \"$(git write-tree)\" = 3bea09bf834de44786c4518651ef1a671042d8d5 || (echo WRONG CODE; exit 1)")
sh("git -c user.name=kaggle -c user.email=kaggle@local commit -qm 'code 80b647ef079f8a650c60e6ddd3bd9bb6f14ed7c8' && echo CODE OK 80b647e")
sh("pip install -q -r requirements-v03-retrieval.txt")
sh("pip install -q -r requirements-v03-generation.txt")
sh("pip install -q --force-reinstall --no-deps torchaudio==2.11.0")
sh("python -c \"import torch, torchaudio, torchvision, vllm, transformers; print(torch.__version__, torch.version.cuda, torchaudio.__version__, torchvision.__version__, vllm.__version__, transformers.__version__); assert torch.cuda.is_available() and torchaudio.__version__.endswith(torch.version.cuda.replace(\x27.\x27, \x27\x27)), \x27torch/torchaudio CUDA mismatch\x27\"")

sh("python -m pytest -q -p no:cacheprovider 2>&1 | tail -3")

In [ ]:
# Cell 2 — restore the v0.2 S1 output FIRST (sub-corpus, joined queries, main runs R0-R6, full-corpus BM25 run),
# then environment record, then NQ corpus text + qrels at the v0.2 pinned revisions
sh("python -m src.restore --name main_s1_results")
sh("ls results/raw/runs/ | grep -c __full__ && ls results/raw/runs/R0__full__*.jsonl")
sh("python -m src.check_env --lock requirements-generation.lock --report-only results/raw/check_env_v04_S5nq.json")
sh("python -m src.env_report --session v04_S5nq --out results/raw/env_v04_S5nq.json")
sh("python -m src.data.download --config configs/v03_nq_qwen.yaml --only nq_corpus,nq_qrels")

In [ ]:
# Cell 3 — in parallel (~1.5 h): encode the FULL NQ corpus (2,681,468 passages); GPU 0 = e5-base-v2, GPU 1 = MiniLM-L6, bge-small
soft("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.encode --config configs/v03_nq_qwen.yaml --models R5 --corpus full",
     "CUDA_VISIBLE_DEVICES=1 python -m src.retrieval.encode --config configs/v03_nq_qwen.yaml --models R1,R2 --corpus full", par=True)

In [ ]:
# Cell 4 — CPU: exact search on the full corpus (all judged queries, top 100), then free the disk; passage-text checks
soft(*[f"python -m src.retrieval.search --config configs/v03_nq_qwen.yaml --model {m} --corpus full --queries judged" for m in ("R1", "R2", "R5")])
sh("ls -la results/raw/runs/ | grep __full__")
sh("rm -rf data/embeddings; df -h /kaggle/working")
soft("python -m src.analysis.passage_answers --config configs/v03_nq_qwen.yaml --runs main:R0,R1,R2,R3,R4,R5,R6 --runs full:R0,R1,R2,R5 "
     "--n 2255 --k 5 --audit R1,R5 --e12 500 --out results/v04")

In [ ]:
# Cell 5 — readers on the full-corpus runs, in parallel (~1.5-2 h): GPU 0 = Qwen, GPU 1 = Llama; R0, R1, R2, R5; 2,255 queries; k = 5
soft("CUDA_VISIBLE_DEVICES=0 python -m src.generation.run --config configs/v03_nq_qwen.yaml --conditions R0,R1,R2,R5 --corpus full --n 2255 --k 5 --auto-eager-retry",
     "CUDA_VISIBLE_DEVICES=1 python -m src.generation.run --config configs/v03_nq_llama.yaml --conditions R0,R1,R2,R5 --corpus full --n 2255 --k 5 --auto-eager-retry",
     par=True)

In [ ]:
# Cell 6 — score answers (EM, F1, answer containment)
soft("python -m src.evaluation.answer --config configs/v03_nq_qwen.yaml --prefix v03_nq_qwen",
     "python -m src.evaluation.answer --config configs/v03_nq_llama.yaml --prefix v03_nq_llama")

In [ ]:
# Last cell — package: everything EXCEPT the large run files (they stay in the earlier output) -> v04_s5nq_results.zip
# small (about 30-60 MB): send this file to Claude
import zipfile
from pathlib import Path
n = 0
with zipfile.ZipFile("/kaggle/working/v04_s5nq_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k in ['results/raw', 'results/processed', 'results/v04', 'configs/resolved_revisions.json', 'logs']:
        for p in ([Path(k)] if Path(k).is_file() else sorted(Path(k).rglob("*"))):
            if p.is_file() and ("/runs/" not in p.as_posix() or p.name.startswith(('R0__full', 'R1__full', 'R2__full', 'R5__full'))) and (not k.startswith("data/") or p.name.startswith(())):
                z.write(p); n += 1
print(n, "files"); sh("ls -la /kaggle/working/v04_s5nq_results.zip")
if GEN_ERR:
    raise RuntimeError("FAILED (packaged anyway): " + GEN_ERR)